# 01 - Export protein and molecule encoders to ONNX 🚀

The default lightweight profile exports NVIDIA ESM-2 35M and IBM MoLFormer 47M. Set `ENCODER_PROFILE = 'legacy'` to export ProLLaMA and Mol-LLaMA instead.

Each encoder is exported without its masked-language-model prediction head, checked against PyTorch, and uploaded to a separate Hugging Face model repository.

In [ ]:
WK_DIR = "/content/"
ENCODER_PROFILE = 'lightweight'  # lightweight or legacy

In [ ]:
%cd {WK_DIR}
!test -d Protein-Ligand-Affinity-Prediction-Using-LLM || git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
%pip install -U pip setuptools wheel ninja packaging
%pip install -e ".[export]"

In [ ]:
import subprocess, sys, torch
if ENCODER_PROFILE == 'legacy':
    version = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
    tag = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
    url = f'https://data.pyg.org/whl/torch-{version}+{tag}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', url])
    print('Restart the runtime after installing legacy graph dependencies.')
else:
    print('Lightweight profile: legacy graph dependencies are not needed.')

## Configuration

In [ ]:
print('Encoder profile:', ENCODER_PROFILE)

In [ ]:
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
from google.colab import userdata
from huggingface_hub import HfApi, login
from pathlib import Path
import os, subprocess
sys.path.insert(0,f"{WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM/src")
HF_USER = userdata.get('HF_USER')
PROTEIN_MODEL_ID = 'facebook/esm2_t12_35M_UR50D'
MOLECULE_MODEL_ID = 'ibm-research/MoLFormer-XL-both-10pct'
PROTEIN_REPO = f'{HF_USER}/esm2-affinity-onnx'
MOLECULE_REPO = f'{HF_USER}/molformer-affinity-onnx'
if ENCODER_PROFILE == 'legacy':
    PROTEIN_MODEL_ID = 'GreatCaptainNemo/ProLLaMA'
    MOLECULE_MODEL_ID = 'DongkiKim/Mol-Llama-3.1-8B-Instruct'
    PROTEIN_REPO = f'{HF_USER}/prollama-affinity-onnx'
    MOLECULE_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
ROOT = Path(f'{WK_DIR}/protein_affinity/onnx')
PROTEIN_ONNX = ROOT / ('esm2' if ENCODER_PROFILE == 'lightweight' else 'prollama')
MOLECULE_ONNX = ROOT / ('molformer' if ENCODER_PROFILE == 'lightweight' else 'mol_llama')
PROTEIN_ONNX.mkdir(parents=True, exist_ok=True)
MOLECULE_ONNX.mkdir(parents=True, exist_ok=True)
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)

## Export the protein encoder

In [ ]:
if ENCODER_PROFILE == 'lightweight':
    target = PROTEIN_ONNX / 'esm2_encoder.onnx'
    command = [
        'affinity-export-esm2-onnx',
        '--model-id', PROTEIN_MODEL_ID,
        '--output', str(PROTEIN_ONNX),
    ]
else:
    target = PROTEIN_ONNX / 'prollama_encoder.onnx'
    command = ['affinity-export-protein-onnx', '--model-id', PROTEIN_MODEL_ID, '--output', str(PROTEIN_ONNX), '--dtype', 'float32', '--sequence-length', '512', '--skip-parity-check']
if not target.exists():
    subprocess.run(command, check=True)
else:
    print('Protein export already exists:', target)

## Export the molecule encoder

In [ ]:
if ENCODER_PROFILE == 'lightweight':
    target = MOLECULE_ONNX / 'molformer_encoder.onnx'
    command = ['affinity-export-molformer-onnx', '--model-id', MOLECULE_MODEL_ID, '--output', str(MOLECULE_ONNX)]
else:
    source = Path('external/Mol-LLaMA')
    if not source.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DongkiKim95/Mol-LLaMA', str(source)], check=True)
        requirements = (source / 'requirements.txt').read_text().replace('rdkit-pypi', 'rdkit').splitlines()
        blocked = ('torch==', 'torchvision==', 'torchaudio==', 'torch-geometric', 'torch_scatter', 'torch-scatter', 'peft==')
        filtered = [line for line in requirements if not line.startswith(blocked)]
        requirements_file = Path('/tmp/mol_llama_requirements.txt')
        requirements_file.write_text('\n'.join(filtered))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'peft>=0.17,<1', 'openbabel-wheel', 'ogb', 'torch-geometric'], check=True)
    target = MOLECULE_ONNX / 'mol_llama_encoder.onnx'
    command = ['affinity-export-mol-onnx', '--official-repo', str(source), '--model-id', MOLECULE_MODEL_ID, '--output', str(MOLECULE_ONNX)]
if not target.exists():
    subprocess.run(command, check=True)
else:
    print('Molecule export already exists:', target)

## ONNX smoke test

In [ ]:
RUN_SMOKE_TEST = False  # Restart the Kaggle session before changing this to True.
if RUN_SMOKE_TEST:
    from affinity.onnx_embeddings import create_onnx_embedder
    protein_type = 'esm2' if ENCODER_PROFILE == 'lightweight' else 'prollama'
    molecule_type = 'molformer' if ENCODER_PROFILE == 'lightweight' else 'mol_llama'
    protein = create_onnx_embedder(protein_type, PROTEIN_ONNX, PROTEIN_MODEL_ID, 1024).encode(['ACDEFGHIKLMNPQRSTVWY'])
    molecule = create_onnx_embedder(molecule_type, MOLECULE_ONNX, MOLECULE_MODEL_ID, 202).encode(['CC(=O)O'])
    print('Protein embedding:', protein.shape)
    print('Molecule embedding:', molecule.shape)
else:
    print('Smoke test skipped. Upload the artifacts, restart, then run validation.')

## Upload both encoder repositories

In [ ]:
import json
import shutil
import onnx
from onnx import TensorProto

UPLOAD = True
CLEAN_REPOSITORIES = True  # Deletes remote repositories and their history.
PRIVATE_REPOSITORIES = False

def validate_onnx_folder(folder):
    folder = Path(folder)
    if ENCODER_PROFILE == 'legacy' and folder == PROTEIN_ONNX:
        metadata_path = folder / 'export_metadata.json'
        metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
        legacy_int8 = folder / 'prollama_encoder_int8.onnx'
        if legacy_int8.exists():
            raise RuntimeError(
                'Rejected legacy INT8 graph is still present. Delete it and its unique '
                'external data, or rerun the legacy exporter with quantization enabled.'
            )
        accepted_int8 = folder / 'int8' / 'prollama_encoder_int8.onnx'
        if accepted_int8.exists() and metadata.get('quantization_status') != 'accepted':
            raise RuntimeError('INT8 graph exists without accepted parity metadata')
    graphs = sorted(folder.rglob('*.onnx'))
    if not graphs:
        raise FileNotFoundError(f'No ONNX graph found in {folder}')
    missing = []
    for graph in graphs:
        model = onnx.load(str(graph), load_external_data=False)
        for tensor in model.graph.initializer:
            if tensor.data_location != TensorProto.EXTERNAL:
                continue
            fields = {item.key: item.value for item in tensor.external_data}
            location = fields.get('location')
            if location and not (graph.parent / location).is_file():
                missing.append(str(graph.parent / location))
    if missing:
        preview = '\n'.join(missing[:10])
        raise FileNotFoundError(f'Missing ONNX external data files:\n{preview}')
    files = [path for path in folder.rglob('*') if path.is_file()]
    size_gib = sum(path.stat().st_size for path in files) / 2**30
    print(f'Validated {folder}: {len(graphs)} graph(s), {len(files)} files, {size_gib:.2f} GiB')

if UPLOAD:
    api = HfApi(token=token)
    uploads = [(PROTEIN_REPO, PROTEIN_ONNX), (MOLECULE_REPO, MOLECULE_ONNX)]

    # Validate every local bundle before deleting anything remotely.
    for _, folder in uploads:
        validate_onnx_folder(folder)

    for repo_id, folder in uploads:
        if CLEAN_REPOSITORIES:
            print('Deleting existing repository:', repo_id)
            api.delete_repo(repo_id, repo_type='model', missing_ok=True)
        api.create_repo(
            repo_id,
            repo_type='model',
            private=PRIVATE_REPOSITORIES,
            exist_ok=True,
        )
        shutil.rmtree(Path(folder) / '.cache' / '.huggingface', ignore_errors=True)
        api.upload_large_folder(
            repo_id=repo_id,
            repo_type='model',
            folder_path=str(folder),
        )
        print('Uploaded clean repository:', repo_id)